In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

In [2]:
df = pd.read_csv('../data/processed/master_entrenamiento.csv')

In [3]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,rank_home,points_home,rank_away,points_away,points_diff,score_diff
0,1993-01-01,Ghana,Mali,1.0,1.0,Friendly,Libreville,Gabon,True,39.0,34.0,69.0,22.0,12.0,0.0
1,1993-01-02,Gabon,Burkina Faso,1.0,1.0,Friendly,Libreville,Gabon,False,55.0,27.0,97.0,11.0,16.0,0.0
2,1993-01-02,Kuwait,Lebanon,2.0,0.0,Friendly,Kuwait City,Kuwait,False,71.0,21.0,161.0,0.0,21.0,2.0
3,1993-01-03,Burkina Faso,Mali,1.0,0.0,Friendly,Libreville,Gabon,True,97.0,11.0,69.0,22.0,-11.0,1.0
4,1993-01-03,Gabon,Ghana,2.0,3.0,Friendly,Libreville,Gabon,False,55.0,27.0,39.0,34.0,-7.0,-1.0


In [4]:
df_oficial = df[df['neutral'] == False]
df_neutral = df[df['neutral'] == True]

In [5]:
promedio_global_local = df_oficial['home_score'].mean()
promedio_global_visitante = df_oficial['away_score'].mean()

goles_totales_neutrales = df_neutral['home_score'].sum() + df_neutral['away_score'].sum()
promedio_global_neutral = goles_totales_neutrales / (2 * len(df_neutral))

print(f'Promedio global local: {promedio_global_local:.4f}')
print(f'Promedio global visitante: {promedio_global_visitante:.4f}')
print(f'Promedio global neutral: {promedio_global_neutral:.4f}')

Promedio global local: 1.6581
Promedio global visitante: 1.0015
Promedio global neutral: 1.3544


In [6]:
stats_local = df_oficial.groupby('home_team').agg(
    goles_anotados_local=('home_score', 'mean'),
    goles_recibidos_local=('away_score', 'mean')
)
stats_visitante = df_oficial.groupby('away_team').agg(
    goles_anotados_visitante=('away_score', 'mean'),
    goles_recibidos_visitante=('home_score', 'mean')
)

stats_local['FA_local'] = stats_local['goles_anotados_local'] / promedio_global_local
stats_local['FD_local'] = stats_local['goles_recibidos_local'] / promedio_global_visitante

stats_visitante['FA_visitante'] = stats_visitante['goles_anotados_visitante'] / promedio_global_visitante
stats_visitante['FD_visitante'] = stats_visitante['goles_recibidos_visitante'] / promedio_global_local

stats_local.fillna(1.0, inplace=True)
stats_visitante.fillna(1.0, inplace=True)

,goles_anotados_visitante,goles_recibidos_visitante,FA_visitante,FD_visitante
away_team,,,,
Afghanistan,0.600000,2.160000,0.599087,1.302663
Albania,0.825000,1.508333,0.823744,0.909653
Algeria,1.134615,1.125000,1.132888,0.678470
American Samoa,0.200000,7.600000,0.199696,4.583445
Andorra,0.277228,2.801980,0.276806,1.689832
...,...,...,...,...
Vietnam,1.111111,1.833333,1.109420,1.105656
Wales,1.017391,1.417391,1.015843,0.854807
Yemen,0.546875,2.062500,0.546043,1.243863


In [7]:
def calcular_fuerza(equipo_A, equipo_B, stats_local, stats_visitante,
                     lam_local, lam_visitante, lam_neutral, es_neutral=False):

    if es_neutral:

        fa_A = (stats_local.loc[equipo_A, 'FA_local'] + stats_visitante.loc[equipo_A, 'FA_visitante']) / 2
        fd_B = (stats_local.loc[equipo_B, 'FD_local'] + stats_visitante.loc[equipo_B, 'FD_visitante']) / 2
        lambda_A = fa_A * fd_B * lam_neutral

        fa_B = (stats_visitante.loc[equipo_B, 'FA_visitante'] + stats_local.loc[equipo_B, 'FA_local']) / 2
        fd_A = (stats_local.loc[equipo_A, 'FD_local'] + stats_visitante.loc[equipo_A, 'FD_visitante']) / 2
        lambda_B = fa_B * fd_A * lam_neutral

    else:
        fa_A = stats_local.loc[equipo_A, 'FA_local']
        fd_B = stats_visitante.loc[equipo_B, 'FD_visitante']
        lambda_A = fa_A * fd_B * lam_local

        fa_B = stats_visitante.loc[equipo_B, 'FA_visitante']
        fd_A = stats_local.loc[equipo_A, 'FD_local']
        lambda_B = fa_B * fd_A * lam_visitante

    return lambda_A, lambda_B

In [8]:
def generar_matriz_poisson(lambda_A, lambda_B, max_goles=5):

    prob_A = [poisson.pmf(i, lambda_A) for i in range(max_goles + 1)]
    prob_B = [poisson.pmf(j, lambda_B) for j in range(max_goles + 1)]

    matriz_prob = np.outer(prob_A, prob_B)

    goles_b = [f"{i} goles B" for i in range(max_goles + 1)]
    filas_a = [f"{i} goles A" for i in range(max_goles + 1)]
    df_matriz = pd.DataFrame(matriz_prob, columns=goles_b, index=filas_a)

    return df_matriz

In [12]:
equipo_prueba_A = 'Argentina'
equipo_prueba_B = 'Ecuador'

lam_A_1, lam_B_1 = calcular_fuerza(
    equipo_prueba_A, equipo_prueba_B, stats_local, stats_visitante,
    promedio_global_local, promedio_global_visitante, promedio_global_neutral, es_neutral=False
)

lam_A_2, lam_B_2 = calcular_fuerza(
    equipo_prueba_A, equipo_prueba_B, stats_local, stats_visitante,
    promedio_global_local, promedio_global_visitante, promedio_global_neutral, es_neutral=True
)

print("--- Argentina de local ---")
print(f"Lambda {equipo_prueba_A}: {lam_A_1:.4f} | Lambda {equipo_prueba_B}: {lam_B_1:.4f}")
print("\n--- Cancha neutral ---")
print(f"Lambda {equipo_prueba_A}: {lam_A_2:.4f} | Lambda {equipo_prueba_B}: {lam_B_2:.4f}")

--- Argentina de local ---
Lambda Argentina: 1.8940 | Lambda Ecuador: 0.5129

--- Cancha neutral ---
Lambda Argentina: 1.4593 | Lambda Ecuador: 0.8138


In [13]:
matriz1 = generar_matriz_poisson(lam_A_1, lam_B_1)
display(matriz1.style.format("{:.2%}"))

,0 goles B,1 goles B,2 goles B,3 goles B,4 goles B,5 goles B
0 goles A,9.01%,4.62%,1.19%,0.20%,0.03%,0.00%
1 goles A,17.06%,8.75%,2.24%,0.38%,0.05%,0.01%
2 goles A,16.16%,8.29%,2.13%,0.36%,0.05%,0.00%
3 goles A,10.20%,5.23%,1.34%,0.23%,0.03%,0.00%
4 goles A,4.83%,2.48%,0.64%,0.11%,0.01%,0.00%
5 goles A,1.83%,0.94%,0.24%,0.04%,0.01%,0.00%


In [14]:
matriz2 = generar_matriz_poisson(lam_A_2, lam_B_2)
display(matriz2.style.format("{:.2%}"))

,0 goles B,1 goles B,2 goles B,3 goles B,4 goles B,5 goles B
0 goles A,10.30%,8.38%,3.41%,0.93%,0.19%,0.03%
1 goles A,15.03%,12.23%,4.98%,1.35%,0.27%,0.04%
2 goles A,10.97%,8.92%,3.63%,0.98%,0.20%,0.03%
3 goles A,5.33%,4.34%,1.77%,0.48%,0.10%,0.02%
4 goles A,1.95%,1.58%,0.64%,0.17%,0.04%,0.01%
5 goles A,0.57%,0.46%,0.19%,0.05%,0.01%,0.00%
